# Notebook 5: Minimal Custom Model Experiment

**What:** Train a simplified 2-source separator (e.g., "voice" vs "noise") from scratch.

**Why:** Practice the full pipeline: data → model → train → inference. Foundation for your own audio model.

**How:** Synthetic dataset, tiny Demucs-style U-Net, end-to-end loop.

## 1. Design Choices (What-Why-How)

| Choice | What | Why |
|--------|------|-----|
| 2 sources | voice, noise | Simpler than 4; faster iteration |
| Synthetic data | Random tones + noise | No dataset download; reproducible |
| Tiny model | channels=16, depth=3 | Fits 3070 Ti, overfits quickly |
| L1 loss | | Standard in Demucs |

In [ ]:
import sys
sys.path.insert(0, r'D:\demucs')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## 2. Synthetic Dataset

Source A: sum of sinusoids ("voice-like")
Source B: filtered noise
Mix = A + B

In [ ]:
def make_synthetic_track(segment_samples, sr=44100):
    t = np.linspace(0, segment_samples/sr, segment_samples, dtype=np.float32)
    # Source A: harmonics (voice-like)
    f0 = 200 + np.random.rand() * 200
    a = 0.3 * (np.sin(2*np.pi*f0*t) + 0.5*np.sin(2*np.pi*2*f0*t))
    # Source B: noise
    b = 0.2 * np.random.randn(segment_samples).astype(np.float32)
    # Stereo: duplicate
    src_a = np.stack([a, a], axis=0)
    src_b = np.stack([b, b], axis=0)
    mix = src_a + src_b
    return (
        torch.from_numpy(mix),
        torch.from_numpy(np.stack([src_a, src_b], axis=0))
    )

sr = 44100
segment_sec = 2
segment_samples = int(sr * segment_sec)
mix, sources = make_synthetic_track(segment_samples)
print(f"Mix: {mix.shape}, Sources: {sources.shape}")

## 3. Minimal Demucs-Style Model

Use the real Demucs with 2 sources and small config.

In [ ]:
from demucs.demucs import Demucs

model = Demucs(
    sources=['voice', 'noise'],
    audio_channels=2,
    channels=16,
    depth=3,
    kernel_size=8,
    stride=4,
    segment=segment_sec,
)

print(f"Params: {sum(p.numel() for p in model.parameters())/1e3:.1f}K")

## 4. Training Loop

Overfit on synthetic data in a few steps.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

batch_size = 8
n_steps = 100

for step in range(n_steps):
    mixes, targets = [], []
    for _ in range(batch_size):
        m, s = make_synthetic_track(segment_samples)
        mixes.append(m)
        targets.append(s)
    mix_batch = torch.stack(mixes).to(device)
    target_batch = torch.stack(targets).to(device)

    pred = model(mix_batch)
    loss = F.l1_loss(pred, target_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"Step {step}, loss={loss.item():.4f}")

print("Done!")

## 5. Inference Check

Separate a new mix and verify output shapes.

In [ ]:
model.eval()
with torch.no_grad():
    mix_test, _ = make_synthetic_track(segment_samples)
    mix_test = mix_test.unsqueeze(0).to(device)
    out = model(mix_test)

print(f"Input: {mix_test.shape}")
print(f"Output: {out.shape} (batch, sources, channels, samples)")
print(f"Sources: {model.sources}")

## 6. Extend to Your Own Model

Ideas:
- **More sources:** Add `guitar`, `piano` (need labeled data)
- **Different task:** Denoising, dereverberation
- **Architecture:** Change `channels`, `depth`, add LSTM (`lstm_layers=1`)
- **Real data:** Use MusDB or your own (mix, stems) pairs

Reference: `demucs/demucs.py` (Demucs), `demucs/hdemucs.py` (HDemucs), `demucs/htdemucs.py` (HTDemucs)